In [1]:
import pandas as pd

# Load the local sample file
df = pd.read_csv("sample30.csv")

# Display the structure to map out the Knowledge Graph triplets
print(f"Loaded {len(df)} rows from sample30.csv.")
print("\nColumns available:")
print(df.columns.tolist())

# Preview the first row to see the data format
print("\nFirst row preview:")
print(df.iloc[0].to_dict())

Loaded 30000 rows from sample30.csv.

Columns available:
['id', 'brand', 'categories', 'manufacturer', 'name', 'reviews_date', 'reviews_didPurchase', 'reviews_doRecommend', 'reviews_rating', 'reviews_text', 'reviews_title', 'reviews_userCity', 'reviews_userProvince', 'reviews_username', 'user_sentiment']

First row preview:
{'id': 'AV13O1A8GV-KLJ3akUyj', 'brand': 'Universal Music', 'categories': 'Movies, Music & Books,Music,R&b,Movies & TV,Movie Bundles & Collections,CDs & Vinyl,Rap & Hip-Hop,Bass,Music on CD or Vinyl,Rap,Hip-Hop,Mainstream Rap,Pop Rap', 'manufacturer': 'Universal Music Group / Cash Money', 'name': 'Pink Friday: Roman Reloaded Re-Up (w/dvd)', 'reviews_date': '2012-11-30T06:21:45.000Z', 'reviews_didPurchase': nan, 'reviews_doRecommend': nan, 'reviews_rating': 5, 'reviews_text': "i love this album. it's very good. more to the hip hop side than her current pop sound.. SO HYPE! i listen to this everyday at the gym! i give it 5star rating all the way. her metaphors are just

In [2]:
df = df.dropna(subset=["id", "reviews_username", "brand", "categories"])

In [3]:
df.isna().sum()

id                          0
brand                       0
categories                  0
manufacturer              141
name                        0
reviews_date               40
reviews_didPurchase     14006
reviews_doRecommend      2541
reviews_rating              0
reviews_text                0
reviews_title             189
reviews_userCity        28037
reviews_userProvince    29770
reviews_username            0
user_sentiment              1
dtype: int64

In [1]:
import pandas as pd

# 1. Load the local data
df = pd.read_csv("sample30.csv")

# Clean rows missing critical structural data
df = df.dropna(subset=["id", "reviews_username", "brand", "categories"])

triplets = []

print("Extracting Knowledge Graph triplets...")
for _, row in df.iterrows():
    item_id = row["id"]

    # 2. User-Item Interaction
    triplets.append((row["reviews_username"], "REVIEWED", item_id))

    # 3. Item-Brand Relation
    triplets.append((item_id, "HAS_BRAND", row["brand"]))

    # 4. Item-Category Relation (Parsing the comma-separated list)
    categories = str(row["categories"]).split(",")
    for cat in categories:
        triplets.append((item_id, "BELONGS_TO", cat.strip()))

# Convert to DataFrame
kg_df = pd.DataFrame(triplets, columns=["Head", "Relation", "Tail"])

# 5. Map entities and relations to integer IDs for Keras Embedding Layers
entities = pd.concat([kg_df["Head"], kg_df["Tail"]]).unique()
relations = kg_df["Relation"].unique()

entity2idx = {ent: i for i, ent in enumerate(entities)}
relation2idx = {rel: i for i, rel in enumerate(relations)}

kg_df["head_idx"] = kg_df["Head"].map(entity2idx)
kg_df["rel_idx"] = kg_df["Relation"].map(relation2idx)
kg_df["tail_idx"] = kg_df["Tail"].map(entity2idx)

print(f"Total Unique Entities: {len(entities)}")
print(f"Total Unique Relations: {len(relations)}")
print(f"Total Graph Triplets Generated: {len(kg_df)}")
print("\nSample Triplets:")
print(kg_df[["Head", "Relation", "Tail"]].head(10))

Extracting Knowledge Graph triplets...
Total Unique Entities: 26722
Total Unique Relations: 3
Total Graph Triplets Generated: 553397

Sample Triplets:
                   Head    Relation                         Tail
0                joshua    REVIEWED         AV13O1A8GV-KLJ3akUyj
1  AV13O1A8GV-KLJ3akUyj   HAS_BRAND              Universal Music
2  AV13O1A8GV-KLJ3akUyj  BELONGS_TO                       Movies
3  AV13O1A8GV-KLJ3akUyj  BELONGS_TO                Music & Books
4  AV13O1A8GV-KLJ3akUyj  BELONGS_TO                        Music
5  AV13O1A8GV-KLJ3akUyj  BELONGS_TO                          R&b
6  AV13O1A8GV-KLJ3akUyj  BELONGS_TO                  Movies & TV
7  AV13O1A8GV-KLJ3akUyj  BELONGS_TO  Movie Bundles & Collections
8  AV13O1A8GV-KLJ3akUyj  BELONGS_TO                  CDs & Vinyl
9  AV13O1A8GV-KLJ3akUyj  BELONGS_TO                Rap & Hip-Hop


In [2]:
df

,id,brand,categories,manufacturer,name,reviews_date,reviews_didPurchase,reviews_doRecommend,reviews_rating,reviews_text,reviews_title,reviews_userCity,reviews_userProvince,reviews_username,user_sentiment
0,AV13O1A8GV-KLJ3akUyj,Universal Music,"Movies, Music & Books,Music,R&b,Movies & TV,Mo...",Universal Music Group / Cash Money,Pink Friday: Roman Reloaded Re-Up (w/dvd),2012-11-30T06:21:45.000Z,NaN,NaN,5,i love this album. it's very good. more to the...,Just Awesome,Los Angeles,NaN,joshua,Positive
1,AV14LG0R-jtxr-f38QfS,Lundberg,"Food,Packaged Foods,Snacks,Crackers,Snacks, Co...",Lundberg,Lundberg Organic Cinnamon Toast Rice Cakes,2017-07-09T00:00:00.000Z,True,NaN,5,Good flavor. This review was collected as part...,Good,NaN,NaN,dorothy w,Positive
2,AV14LG0R-jtxr-f38QfS,Lundberg,"Food,Packaged Foods,Snacks,Crackers,Snacks, Co...",Lundberg,Lundberg Organic Cinnamon Toast Rice Cakes,2017-07-09T00:00:00.000Z,True,NaN,5,Good flavor.,Good,NaN,NaN,dorothy w,Positive
3,AV16khLE-jtxr-f38VFn,K-Y,"Personal Care,Medicine Cabinet,Lubricant/Sperm...",K-Y,K-Y Love Sensuality Pleasure Gel,2016-01-06T00:00:00.000Z,False,False,1,I read through the reviews on here before look...,Disappointed,NaN,NaN,rebecca,Negative
4,AV16khLE-jtxr-f38VFn,K-Y,"Personal Care,Medicine Cabinet,Lubricant/Sperm...",K-Y,K-Y Love Sensuality Pleasure Gel,2016-12-21T00:00:00.000Z,False,False,1,My husband bought this gel for us. The gel cau...,Irritation,NaN,NaN,walker557,Negative
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,AVpfW8y_LJeJML437ySW,L'oreal Paris,"Beauty,Hair Care,Shampoo & Conditioner,Holiday...",L'oreal Paris,L'or233al Paris Elvive Extraordinary Clay Reba...,2017-01-23T00:00:00.000Z,False,True,5,I got this conditioner with Influenster to try...,Softness!!,NaN,NaN,laurasnchz,Positive
29996,AVpfW8y_LJeJML437ySW,L'oreal Paris,"Beauty,Hair Care,Shampoo & Conditioner,Holiday...",L'oreal Paris,L'or233al Paris Elvive Extraordinary Clay Reba...,2017-01-27T00:00:00.000Z,False,True,5,"I love it , I received this for review purpose...",I love it,NaN,NaN,scarlepadilla,Positive
29997,AVpfW8y_LJeJML437ySW,L'oreal Paris,"Beauty,Hair Care,Shampoo & Conditioner,Holiday...",L'oreal Paris,L'or233al Paris Elvive Extraordinary Clay Reba...,2017-01-21T00:00:00.000Z,False,True,5,First of all I love the smell of this product....,Hair is so smooth after use,NaN,NaN,liviasuexo,Positive
29998,AVpfW8y_LJeJML437ySW,L'oreal Paris,"Beauty,Hair Care,Shampoo & Conditioner,Holiday...",L'oreal Paris,L'or233al Paris Elvive Extraordinary Clay Reba...,2017-01-11T00:00:00.000Z,False,True,5,I received this through Influenster and will n...,Perfect for my oily hair!,NaN,NaN,ktreed95,Positive


In [3]:
entity2idx

{'joshua': 0,
 'AV13O1A8GV-KLJ3akUyj': 1,
 'dorothy w': 2,
 'AV14LG0R-jtxr-f38QfS': 3,
 'rebecca': 4,
 'AV16khLE-jtxr-f38VFn': 5,
 'walker557': 6,
 'samantha': 7,
 'raeanne': 8,
 'kimmie': 9,
 'cassie': 10,
 'moore222': 11,
 'jds1992': 12,
 'bre234': 13,
 'gordy313': 14,
 'nicole': 15,
 'cvperez': 16,
 'beccagrl532': 17,
 'sanchez': 18,
 'll24': 19,
 'browns fan': 20,
 'just faith everyday': 21,
 'vero': 22,
 'jo276': 23,
 'ashley a': 24,
 'jp71': 25,
 'jkell': 26,
 'karen': 27,
 'warren': 28,
 'mas0814': 29,
 'amanda': 30,
 'peach pie': 31,
 'AV1d76w7vKc47QAVhCqn': 32,
 'birdwoman': 33,
 'michelle': 34,
 'jean z': 35,
 'vanessa mcnally': 36,
 '5alarm': 37,
 'danielle': 38,
 'AV1h6gSl-jtxr-f31p40': 39,
 'shedove': 40,
 'cynthiacc': 41,
 'dfwatheartgirl': 42,
 'crissyx5': 43,
 'AV1h6Gu0glJLPUi8IjA_': 44,
 'nyisha m': 45,
 'gardenbunny318': 46,
 'solo': 47,
 'sasparilla': 48,
 'mkris18': 49,
 'momof2': 50,
 'mommyshappy8714': 51,
 'eyo': 52,
 'anonymous8589': 53,
 'jazzymom': 54,
 'ramon

In [4]:
relation2idx

{'REVIEWED': 0, 'HAS_BRAND': 1, 'BELONGS_TO': 2}

In [5]:
triplets = []

print("Extracting Knowledge Graph triplets...")
for _, row in df.iterrows():
    item_id = row["id"]

    # 2. User-Item Interaction
    triplets.append((row["reviews_username"], "REVIEWED", item_id))

    # 3. Item-Brand Relation
    triplets.append((item_id, "HAS_BRAND", row["brand"]))

    # 4. Item-Category Relation (Parsing the comma-separated list)
    categories = str(row["categories"]).split(",")
    for cat in categories:
        triplets.append((item_id, "BELONGS_TO", cat.strip()))

Extracting Knowledge Graph triplets...


In [6]:
triplets

[('joshua', 'REVIEWED', 'AV13O1A8GV-KLJ3akUyj'),
 ('AV13O1A8GV-KLJ3akUyj', 'HAS_BRAND', 'Universal Music'),
 ('AV13O1A8GV-KLJ3akUyj', 'BELONGS_TO', 'Movies'),
 ('AV13O1A8GV-KLJ3akUyj', 'BELONGS_TO', 'Music & Books'),
 ('AV13O1A8GV-KLJ3akUyj', 'BELONGS_TO', 'Music'),
 ('AV13O1A8GV-KLJ3akUyj', 'BELONGS_TO', 'R&b'),
 ('AV13O1A8GV-KLJ3akUyj', 'BELONGS_TO', 'Movies & TV'),
 ('AV13O1A8GV-KLJ3akUyj', 'BELONGS_TO', 'Movie Bundles & Collections'),
 ('AV13O1A8GV-KLJ3akUyj', 'BELONGS_TO', 'CDs & Vinyl'),
 ('AV13O1A8GV-KLJ3akUyj', 'BELONGS_TO', 'Rap & Hip-Hop'),
 ('AV13O1A8GV-KLJ3akUyj', 'BELONGS_TO', 'Bass'),
 ('AV13O1A8GV-KLJ3akUyj', 'BELONGS_TO', 'Music on CD or Vinyl'),
 ('AV13O1A8GV-KLJ3akUyj', 'BELONGS_TO', 'Rap'),
 ('AV13O1A8GV-KLJ3akUyj', 'BELONGS_TO', 'Hip-Hop'),
 ('AV13O1A8GV-KLJ3akUyj', 'BELONGS_TO', 'Mainstream Rap'),
 ('AV13O1A8GV-KLJ3akUyj', 'BELONGS_TO', 'Pop Rap'),
 ('dorothy w', 'REVIEWED', 'AV14LG0R-jtxr-f38QfS'),
 ('AV14LG0R-jtxr-f38QfS', 'HAS_BRAND', 'Lundberg'),
 ('AV14LG0R-jt

In [7]:
kg_df = pd.DataFrame(triplets, columns=["Head", "Relation", "Tail"])

In [8]:
kg_df

,Head,Relation,Tail
0,joshua,REVIEWED,AV13O1A8GV-KLJ3akUyj
1,AV13O1A8GV-KLJ3akUyj,HAS_BRAND,Universal Music
2,AV13O1A8GV-KLJ3akUyj,BELONGS_TO,Movies
3,AV13O1A8GV-KLJ3akUyj,BELONGS_TO,Music & Books
4,AV13O1A8GV-KLJ3akUyj,BELONGS_TO,Music
...,...,...,...
553392,AVpfW8y_LJeJML437ySW,BELONGS_TO,Health & Beauty
553393,AVpfW8y_LJeJML437ySW,BELONGS_TO,L'oreal
553394,AVpfW8y_LJeJML437ySW,BELONGS_TO,Personal Care
553395,AVpfW8y_LJeJML437ySW,BELONGS_TO,Hair Treatments


In [9]:
kg_df[kg_df["Relation"] == "Belongs_to".upper()]

,Head,Relation,Tail
2,AV13O1A8GV-KLJ3akUyj,BELONGS_TO,Movies
3,AV13O1A8GV-KLJ3akUyj,BELONGS_TO,Music & Books
4,AV13O1A8GV-KLJ3akUyj,BELONGS_TO,Music
5,AV13O1A8GV-KLJ3akUyj,BELONGS_TO,R&b
6,AV13O1A8GV-KLJ3akUyj,BELONGS_TO,Movies & TV
...,...,...,...
553392,AVpfW8y_LJeJML437ySW,BELONGS_TO,Health & Beauty
553393,AVpfW8y_LJeJML437ySW,BELONGS_TO,L'oreal
553394,AVpfW8y_LJeJML437ySW,BELONGS_TO,Personal Care
553395,AVpfW8y_LJeJML437ySW,BELONGS_TO,Hair Treatments


In [13]:
# entities and relations to integer IDs for Keras Embedding Layers
entities = pd.concat([kg_df["Head"], kg_df["Tail"]]).unique()
relations = kg_df["Relation"].unique()

entity2idx = {ent: i for i, ent in enumerate(entities)}
relation2idx = {rel: i for i, rel in enumerate(relations)}

kg_df["head_idx"] = kg_df["Head"].map(entity2idx)
kg_df["rel_idx"] = kg_df["Relation"].map(relation2idx)
kg_df["tail_idx"] = kg_df["Tail"].map(entity2idx)

print(f"Total Unique Entities: {len(entities)}")
print(f"Total Unique Relations: {len(relations)}")
print(f"Total Graph Triplets Generated: {len(kg_df)}")
print("\nSample Triplets:")
kg_df[["Head", "Relation", "Tail"]].head(10)

Total Unique Entities: 26722
Total Unique Relations: 3
Total Graph Triplets Generated: 553397

Sample Triplets:


,Head,Relation,Tail
0,joshua,REVIEWED,AV13O1A8GV-KLJ3akUyj
1,AV13O1A8GV-KLJ3akUyj,HAS_BRAND,Universal Music
2,AV13O1A8GV-KLJ3akUyj,BELONGS_TO,Movies
3,AV13O1A8GV-KLJ3akUyj,BELONGS_TO,Music & Books
4,AV13O1A8GV-KLJ3akUyj,BELONGS_TO,Music
5,AV13O1A8GV-KLJ3akUyj,BELONGS_TO,R&b
6,AV13O1A8GV-KLJ3akUyj,BELONGS_TO,Movies & TV
7,AV13O1A8GV-KLJ3akUyj,BELONGS_TO,Movie Bundles & Collections
8,AV13O1A8GV-KLJ3akUyj,BELONGS_TO,CDs & Vinyl
9,AV13O1A8GV-KLJ3akUyj,BELONGS_TO,Rap & Hip-Hop


In [10]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data

# 1. Define the Graph Structure (Edges)
# Edges are represented as a 2D tensor of source and target nodes
edge_index = torch.tensor([[0, 1, 1, 2], [1, 0, 2, 1]], dtype=torch.long)

# 2. Define Node Features
# Each node gets a feature vector (e.g., 3 nodes with 2 features each)
x = torch.tensor([[-1.0, 0.0], [0.0, 1.0], [1.0, -1.0]], dtype=torch.float)

# Create the Graph Data object
data = Data(x=x, edge_index=edge_index)


# 3. Define the GNN Model
class GCN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = GCNConv(2, 16)  # Input features: 2, Output: 16
        self.conv2 = GCNConv(16, 2)  # Input: 16, Output: 2 (classes)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = self.conv1(x, edge_index)
        x = x.relu()
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)


# 4. Instantiate and Run the Model
model = GCN()
out = model(data)
print(out)

tensor([[-0.7333, -0.6546],
        [-0.7152, -0.6716],
        [-0.6833, -0.7031]], grad_fn=<LogSoftmaxBackward0>)


In [11]:
model

GCN(
  (conv1): GCNConv(2, 16)
  (conv2): GCNConv(16, 2)
)

In [13]:
type(data)

torch_geometric.data.data.Data

In [14]:
dir(data)

['__abstractmethods__',
 '__annotate_func__',
 '__call__',
 '__cat_dim__',
 '__class__',
 '__contains__',
 '__copy__',
 '__deepcopy__',
 '__delattr__',
 '__delitem__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattr__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__inc__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__len__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setitem__',
 '__setstate__',
 '__sizeof__',
 '__slots__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_abc_impl',
 '_edge_attr_cls',
 '_edge_to_layout',
 '_edges_to_layout',
 '_find_parent',
 '_get_edge_index',
 '_get_tensor',
 '_get_tensor_size',
 '_multi_get_tensor',
 '_put_edge_index',
 '_put_tensor',
 '_remove_edge_index',
 '_remove_tensor',
 '_store',
 '_tensor_attr_cls',
 '_to_type',
 '_union',
 'apply',

In [17]:
data.to_dict()

{'x': tensor([[-1.,  0.],
         [ 0.,  1.],
         [ 1., -1.]]),
 'edge_index': tensor([[0, 1, 1, 2],
         [1, 0, 2, 1]])}

In [15]:
data.num_edges

4

In [18]:
device = "mps"

In [19]:
model = model.to(device)

In [20]:
data = data.to(device)

In [22]:
model(data)

tensor([[-0.7333, -0.6546],
        [-0.7152, -0.6716],
        [-0.6833, -0.7031]], device='mps:0', grad_fn=<LogSoftmaxBackward0>)

In [10]:
entities = pd.concat([kg_df["Head"], kg_df["Tail"]]).unique()

In [11]:
entities

<ArrowStringArray>
[                   'joshua',      'AV13O1A8GV-KLJ3akUyj',
                 'dorothy w',      'AV14LG0R-jtxr-f38QfS',
                   'rebecca',      'AV16khLE-jtxr-f38VFn',
                 'walker557',                  'samantha',
                   'raeanne',                    'kimmie',
 ...
      'Calendars & Planners',                  'Planners',
    'Calendars and Planners',              'All Planners',
     'Planners & Organizers',             'Time Planners',
          'Monthly Planners',      'Electronics Features',
 'Glass and Surface Cleaner',             'L'oreal Paris']
Length: 26722, dtype: str

In [3]:
# import tensorflow as tf
# from tensorflow.keras import layers, Model
#
# class KnowledgeAwareRecommender(Model):
#     def __init__(self, num_entities, num_relations, embedding_dim=64, **kwargs):
#         super(KnowledgeAwareRecommender, self).__init__(**kwargs)
#
#         # 1. Knowledge Graph Embedding Layers (TransE style)
#         self.entity_embedding = layers.Embedding(
#             input_dim=num_entities,
#             output_dim=embedding_dim,
#             name="entity_embedding_layer"
#         )
#         self.relation_embedding = layers.Embedding(
#             input_dim=num_relations,
#             output_dim=embedding_dim,
#             name="relation_embedding_layer"
#         )
#
#         # 2. Feature Fusion & Dense Layers
#         self.fusion_dense1 = layers.Dense(128, activation='relu', name='fusion_hidden')
#         self.dropout = layers.Dropout(0.2)
#
#         # 3. Multi-Task Output Heads
#         # Task 1: Rating Prediction (Regression: 1 to 5 stars)
#         self.rating_output = layers.Dense(1, activation='linear', name='rating_head')
#
#         # Task 2: Sentiment/Preference Classification (Binary: Positive/Negative)
#         self.sentiment_output = layers.Dense(1, activation='sigmoid', name='sentiment_head')
#
#     def call(self, inputs):
#         # Unpack structural graph inputs: head entity, relation, tail entity
#         head_idx = inputs['head_idx']
#         rel_idx = inputs['rel_idx']
#         tail_idx = inputs['tail_idx']
#
#         # Get vector representations
#         h = self.entity_embedding(head_idx)
#         r = self.relation_embedding(rel_idx)
#         t = self.entity_embedding(tail_idx)
#
#         # TransE Score Calculation: ||h + r - t|| (Lower distance = stronger structural validity)
#         trans_e_score = tf.norm(h + r - t, axis=1, keepdims=True)
#
#         # Fuse structural embeddings for recommendation prediction
#         fusion_input = layers.concatenate([h, r, t, trans_e_score])
#         x = self.fusion_dense1(fusion_input)
#         x = self.dropout(x)
#
#         # Multi-task predictions
#         rating_pred = self.rating_output(x)
#         sentiment_pred = self.sentiment_output(x)
#
#         return {
#             "rating": rating_pred,
#             "sentiment": sentiment_pred,
#             "trans_e_score": trans_e_score
#         }
#
# # Instantiate the model with your exact graph dimensions
# NUM_ENTITIES = 26722
# NUM_RELATIONS = 3
# EMBEDDING_DIM = 64
#
# model = KnowledgeAwareRecommender(
#     num_entities=NUM_ENTITIES,
#     num_relations=NUM_RELATIONS,
#     embedding_dim=EMBEDDING_DIM
# )
#
# model.compile(
#     optimizer='adam',
#     loss={
#         'rating': 'mean_squared_error',
#         'sentiment': 'binary_crossentropy',
#         'trans_e_score': 'mean_absolute_error' # Minimizes structural translation error
#     },
#     loss_weights={
#         'rating': 1.0,
#         'sentiment': 0.5,
#         'trans_e_score': 0.2
#     }, jit_compile= False
# )
#
# print("Knowledge-Aware Multi-Task Model successfully compiled and ready for training!")

I0000 00:00:1789680307.599011    6928 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1789680308.119217    6928 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI AVX_VNNI_INT8 AVX_NE_CONVERT FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1789680310.222159    6928 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Knowledge-Aware Multi-Task Model successfully compiled and ready for training!


W0000 00:00:1789680312.089982    6928 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
W0000 00:00:1789680312.093330    6928 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
I0000 00:00:1789680312.289850    6928 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5262 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 12.0a


In [23]:
kg_df

,Head,Relation,Tail
0,joshua,REVIEWED,AV13O1A8GV-KLJ3akUyj
1,AV13O1A8GV-KLJ3akUyj,HAS_BRAND,Universal Music
2,AV13O1A8GV-KLJ3akUyj,BELONGS_TO,Movies
3,AV13O1A8GV-KLJ3akUyj,BELONGS_TO,Music & Books
4,AV13O1A8GV-KLJ3akUyj,BELONGS_TO,Music
...,...,...,...
553392,AVpfW8y_LJeJML437ySW,BELONGS_TO,Health & Beauty
553393,AVpfW8y_LJeJML437ySW,BELONGS_TO,L'oreal
553394,AVpfW8y_LJeJML437ySW,BELONGS_TO,Personal Care
553395,AVpfW8y_LJeJML437ySW,BELONGS_TO,Hair Treatments


In [24]:
entities

<ArrowStringArray>
[                   'joshua',      'AV13O1A8GV-KLJ3akUyj',
                 'dorothy w',      'AV14LG0R-jtxr-f38QfS',
                   'rebecca',      'AV16khLE-jtxr-f38VFn',
                 'walker557',                  'samantha',
                   'raeanne',                    'kimmie',
 ...
      'Calendars & Planners',                  'Planners',
    'Calendars and Planners',              'All Planners',
     'Planners & Organizers',             'Time Planners',
          'Monthly Planners',      'Electronics Features',
 'Glass and Surface Cleaner',             'L'oreal Paris']
Length: 26722, dtype: str

In [4]:
# import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# # 👇 ADD THESE TWO LINES 👇
# os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=-1"
# os.environ["TF_XLA_DISABLE_XLA_DEVICES_AND_KERNELS"] = "1"
#
# import tensorflow as tf
#
# import numpy as np
# import tensorflow as tf
#
# # 1. Explicitly cast inputs to int32 and targets to float32 to prevent DirectML stalls
# head_inputs = kg_df['head_idx'].values.astype(np.int32)
# rel_inputs = kg_df['rel_idx'].values.astype(np.int32)
# tail_inputs = kg_df['tail_idx'].values.astype(np.int32)
#
# X_train = {
#     'head_idx': head_inputs,
#     'rel_idx': rel_inputs,
#     'tail_idx': tail_inputs
# }
#
# num_samples = len(kg_df)
# y_train = {
#     'rating': np.random.uniform(1.0, 5.0, size=(num_samples, 1)).astype(np.float32),
#     'sentiment': np.random.randint(0, 2, size=(num_samples, 1)).astype(np.float32),
#     'trans_e_score': np.zeros((num_samples, 1), dtype=np.float32)
# }
#
# # 2. Re-run training with clean 32-bit tensors
# print("Starting training with optimized 32-bit tensors...")
# history = model.fit(
#     X_train,
#     y_train,
#     batch_size=256,
#     epochs=3,
#     validation_split=0.1
# )
#
# print("Training cycle complete successfully!")

Starting training with optimized 32-bit tensors...
Epoch 1/3


E0000 00:00:1789680313.023065    7042 ptx_compiler_helpers.cc:154] *** WARNING *** Invoking ptxas with version 12.0.140, which corresponds to a CUDA version <=12.6.2. CUDA versions 12.x.y up to and including 12.6.2 miscompile certain edge cases around clamping.
Please upgrade to CUDA 12.6.3 or newer.
W0000 00:00:1789680313.025075    7042 subprocess_compilation.cc:241] Falling back to the CUDA driver for PTX compilation; ptxas does not support CC 12.0
W0000 00:00:1789680313.025107    7042 subprocess_compilation.cc:244] Used ptxas at /usr/bin/ptxas
W0000 00:00:1789680313.025165    7042 gpu_kernel_to_blob_pass.cc:190] Failed to compile generated PTX with ptxas. Falling back to compilation by driver.
W0000 00:00:1789680313.028010    7053 gpu_kernel_to_blob_pass.cc:190] Failed to compile generated PTX with ptxas. Falling back to compilation by driver.
W0000 00:00:1789680313.029131    7048 gpu_kernel_to_blob_pass.cc:190] Failed to compile generated PTX with ptxas. Falling back to compilation

1945/1946 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 1.7843 - rating_loss: 1.3901 - sentiment_loss: 0.6934 - trans_e_score_loss: 0.2375 

W0000 00:00:1789680352.233914    7272 gpu_kernel_to_blob_pass.cc:190] Failed to compile generated PTX with ptxas. Falling back to compilation by driver.
W0000 00:00:1789680352.235536    7279 gpu_kernel_to_blob_pass.cc:190] Failed to compile generated PTX with ptxas. Falling back to compilation by driver.
W0000 00:00:1789680352.236943    7274 gpu_kernel_to_blob_pass.cc:190] Failed to compile generated PTX with ptxas. Falling back to compilation by driver.
W0000 00:00:1789680352.238966    7280 gpu_kernel_to_blob_pass.cc:190] Failed to compile generated PTX with ptxas. Falling back to compilation by driver.
W0000 00:00:1789680352.240362    7277 gpu_kernel_to_blob_pass.cc:190] Failed to compile generated PTX with ptxas. Falling back to compilation by driver.
W0000 00:00:1789680352.241562    7278 gpu_kernel_to_blob_pass.cc:190] Failed to compile generated PTX with ptxas. Falling back to compilation by driver.
W0000 00:00:1789680352.243380    7276 gpu_kernel_to_blob_pass.cc:190] Failed to co

1946/1946 ━━━━━━━━━━━━━━━━━━━━ 41s 14ms/step - loss: 1.7843 - rating_loss: 1.3901 - sentiment_loss: 0.6934 - trans_e_score_loss: 0.2374 - val_loss: 1.8079 - val_rating_loss: 1.3304 - val_sentiment_loss: 0.6934 - val_trans_e_score_loss: 0.6590
Epoch 2/3
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 23s 12ms/step - loss: 1.6707 - rating_loss: 1.3075 - sentiment_loss: 0.6909 - trans_e_score_loss: 0.0890 - val_loss: 1.8217 - val_rating_loss: 1.3574 - val_sentiment_loss: 0.6988 - val_trans_e_score_loss: 0.5778
Epoch 3/3
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 24s 12ms/step - loss: 1.6423 - rating_loss: 1.2900 - sentiment_loss: 0.6772 - trans_e_score_loss: 0.0682 - val_loss: 1.8260 - val_rating_loss: 1.3690 - val_sentiment_loss: 0.6992 - val_trans_e_score_loss: 0.5415
Training cycle complete successfully!


In [ ]:
import tensorflow as tf
import torch
from tensorflow.keras import layers, Model
import tensorflow as tf

# !pip uninstall -y transformers tensorflow
# !pip install tensorflow transformers
import tensorflow as tf
from transformers import BertModel, BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertModel.from_pretrained("bert-base-uncased")


class BERTKnowledgeAwareRecommender(Model):
    def __init__(self, num_entities, num_relations, embedding_dim=64, **kwargs):
        super(BERTKnowledgeAwareRecommender, self).__init__(**kwargs)

        # 1. Knowledge Graph Embedding Layers (TransE style)
        self.entity_embedding = layers.Embedding(
            input_dim=num_entities,
            output_dim=embedding_dim,
            name="entity_embedding_layer",
        )
        self.relation_embedding = layers.Embedding(
            input_dim=num_relations,
            output_dim=embedding_dim,
            name="relation_embedding_layer",
        )

        # 2. Pre-trained BERT Model for Unstructured Text Reviews
        # Using 'bert-base-uncased' as the foundational encoder
        self.bert = BertModel.from_pretrained("bert-base-uncased")

        # Project BERT's 768-dim output down to match your embedding dimension (e.g., 64)
        self.bert_projection = layers.Dense(
            embedding_dim, activation="relu", name="bert_projection"
        )

        # 3. Fusion & Dense Layers
        # Concatenated inputs: h (64) + r (64) + t (64) + trans_e_score (1) + projected_text (64) = 257 dimensions
        self.fusion_dense = layers.Dense(128, activation="relu", name="fusion_hidden")
        self.dropout = layers.Dropout(0.2)

        # 4. Multi-Task Output Heads
        self.rating_output = layers.Dense(1, activation="linear", name="rating_head")
        self.sentiment_output = layers.Dense(
            1, activation="sigmoid", name="sentiment_head"
        )

    def call(self, inputs):
        # Unpack Graph Inputs
        head_idx = inputs["head_idx"]
        rel_idx = inputs["rel_idx"]
        tail_idx = inputs["tail_idx"]

        h = self.entity_embedding(head_idx)
        r = self.relation_embedding(rel_idx)
        t = self.entity_embedding(tail_idx)

        # TransE Score Calculation: ||h + r - t||
        trans_e_score = tf.norm(h + r - t, axis=1, keepdims=True)

        # Unpack Text Inputs (BERT tokenized tensors)
        input_ids = inputs["input_ids"]
        attention_mask = inputs["attention_mask"]

        # Extract text representations from BERT
        bert_outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = bert_outputs.pooler_output  # [Batch Size, 768]
        text_features = self.bert_projection(
            pooled_output
        )  # [Batch Size, embedding_dim]

        # Fuse Structural Graph Embeddings + TransE Score + BERT Text Features
        fusion_input = layers.concatenate(
            [h, r, t, trans_e_score, text_features], axis=-1
        )
        x = self.fusion_dense(fusion_input)
        x = self.dropout(x)

        # Multi-task predictions
        rating_pred = self.rating_output(x)
        sentiment_pred = self.sentiment_output(x)

        return {
            "rating": rating_pred,
            "sentiment": sentiment_pred,
            "trans_e_score": trans_e_score,
        }


# Instantiate the full hybrid model
NUM_ENTITIES = 26722
NUM_RELATIONS = 3
EMBEDDING_DIM = 64

model = BERTKnowledgeAwareRecommender(
    num_entities=NUM_ENTITIES, num_relations=NUM_RELATIONS, embedding_dim=EMBEDDING_DIM
)

# Compile with multi-task loss configuration
model.compile(
    optimizer="adam",
    loss={
        "rating": "mean_squared_error",
        "sentiment": "binary_crossentropy",
        "trans_e_score": "mean_absolute_error",
    },
    loss_weights={"rating": 1.0, "sentiment": 0.5, "trans_e_score": 0.2},
)

print("Full Hybrid BERT + Knowledge Graph Multi-Task Model successfully compiled!")

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for gpu in gpus:
        details = tf.config.experimental.get_device_details(gpu)
        print("GPU Device Name:", details.get("device_name", "Unknown GPU"))
else:
    print("No GPU found.")

In [ ]:
import os

# 1. Force NVIDIA GPU visibility (bypasses Intel integrated graphics)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# 2. Silence generic CPU and math library optimization warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import tensorflow as tf
import numpy as np

print("=" * 50)
print("🔍 RUNNING WSL2 GPU DIAGNOSTIC")
print("=" * 50)

# Check if any GPU is detected
gpus = tf.config.list_physical_devices("GPU")
if not gpus:
    print("❌ ERROR: No GPU detected! Check your WSL2 NVIDIA drivers.")
    exit()

# Print exact hardware details
details = tf.config.experimental.get_device_details(gpus[0])
print(f"✅ Found GPU: {details.get('device_name', 'Unknown')}")
print(f"✅ Compute Capability: {details.get('compute_capability', 'Unknown')}\n")

print("🚀 Testing a live training loop on the GPU...")
print(
    "💡 (Note: If this laptop has a brand new GPU, it might pause here for a moment to compile kernels)\n"
)

# Create random dummy data (1000 samples, 32 features)
x_train = np.random.random((1000, 32)).astype(np.float32)
y_train = np.random.random((1000, 1)).astype(np.float32)

# Build a tiny 1-layer model
model = tf.keras.Sequential([tf.keras.layers.Dense(1, input_shape=(32,))])

model.compile(optimizer="adam", loss="mse", jit_compile=False)

# Run for 5 quick epochs to verify hardware execution
history = model.fit(x_train, y_train, epochs=5, batch_size=64, verbose=1)

print("\n🎉 SUCCESS! TensorFlow successfully trained on the GPU.")
print("=" * 50)

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# 👇 ADD THESE TWO LINES 👇
os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=-1"
os.environ["TF_XLA_DISABLE_XLA_DEVICES_AND_KERNELS"] = "1"

import tensorflow as tf

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

# Search for models matching 'bert', designed for 'tensorflow', sorted by downloads
models = api.list_models(
    filter="tensorflow",
    search="bert",
    sort="downloads",
    limit=20,  # Adjust the limit to see more or fewer models
)

print("=" * 60)
print("📚 AVAILABLE TENSORFLOW BERT MODELS ON HUGGING FACE")
print("=" * 60)

for model in models:
    # Print the model ID and its download count
    print(f"🔹 {model.modelId:<35} | 💾 Downloads: {model.downloads:,}")